In [ ]:
#Created with the help of ChatGPT

In [ ]:
import polars as pl
import requests

# 10 Beispiel-Standorte (Koordinaten für Deutschland)
lats = [52.52, 48.13, 53.55, 50.11, 51.22, 48.77, 51.34, 51.05, 52.37, 51.26]
lons = [13.41, 11.57, 9.99, 8.68, 6.77, 9.18, 12.37, 13.73, 9.73, 7.11]

params = {
    "latitude": lats,
    "longitude": lons,
    "start_date": "2014-01-01",
    "end_date": "2024-01-01",
    "hourly": ["temperature_2m", "precipitation", "weather_code"],
    "timezone": "UTC"
}

In [2]:
print("Sende API-Call für 10 Standorte...")
response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)

if response.status_code == 200:
    data = response.json()
    if not isinstance(data, list): data = [data] # Sicherstellen, dass es eine Liste ist
    
    # Daten in Polars DataFrames umwandeln
    frames = []
    for entry in data:
        df_loc = pl.DataFrame(entry["hourly"]).with_columns([
            pl.lit(entry["latitude"]).alias("lat").cast(pl.Float32),
            pl.lit(entry["longitude"]).alias("lon").cast(pl.Float32),
            pl.col("time").str.to_datetime(),
            pl.col("weather_code").cast(pl.UInt8),
            pl.col("temperature_2m").cast(pl.Float32),
            pl.col("precipitation").cast(pl.Float32)
        ])
        frames.append(df_loc)
    
    # Zu einem großen DataFrame verbinden
    df = pl.concat(frames)
    print(f"Erfolg! {df.height:,} Zeilen geladen.")
else:
    print(f"Fehler: {response.status_code}")

Sende API-Call für 10 Standorte...
Erfolg! 876,720 Zeilen geladen.


In [3]:
# Als Parquet speichern (zstd Kompression ist Standard und sehr effizient)
df.write_parquet("test_weather_data.parquet")

print("Datei 'test_weather_data.parquet' wurde erstellt.")

# Kurze Vorschau der Daten und Datentypen
print(df.schema)
display(df.head())

Datei 'test_weather_data.parquet' wurde erstellt.
Schema({'time': Datetime(time_unit='us', time_zone=None), 'temperature_2m': Float32, 'precipitation': Float32, 'weather_code': UInt8, 'lat': Float32, 'lon': Float32})


time,temperature_2m,precipitation,weather_code,lat,lon
datetime[μs],f32,f32,u8,f32,f32
2014-01-01 00:00:00,-0.3,0.0,3,52.548328,13.407822
2014-01-01 01:00:00,-0.2,0.0,3,52.548328,13.407822
2014-01-01 02:00:00,-0.1,0.0,3,52.548328,13.407822
2014-01-01 03:00:00,0.2,0.0,3,52.548328,13.407822
2014-01-01 04:00:00,0.3,0.0,3,52.548328,13.407822


In [4]:
# Die Daten aus dem DataFrame als CSV speichern
df.write_csv("test_weather_data.csv")

print("Datei 'test_weather_data.csv' wurde erfolgreich erstellt.")

Datei 'test_weather_data.csv' wurde erfolgreich erstellt.
